**Content Produced by UF Signal Processing Society**

**Authors: Raul Valle & Contributors**

# Audio & Speech DSP

The most tangible application of everything in the DSP track: sound. We synthesize, analyze, and mangle audio with the tools you already own — spectrograms, filters, and source-filter models. Every cell produces a signal you can export and *listen to* (`scipy.io.wavfile.write`; in Jupyter, `IPython.display.Audio(x, rate=fs)`).

## 1. Pre-requisites

- [Foundations of Signal Processing 1](./Foundations_of_Signal_Processing_1.ipynb) S6 (STFT).
- [Filter Design](./Filter_Design.ipynb).

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal as sig
rng = np.random.default_rng(0)
fs = 16_000

def show_spec(x, title="", fs=fs):
    f, t, S = sig.stft(x, fs=fs, nperseg=512)
    plt.figure(figsize=(8.5, 2.8))
    plt.pcolormesh(t, f, 20*np.log10(np.abs(S) + 1e-8), shading="auto", vmin=-100, vmax=-20)
    plt.ylabel("Hz"); plt.xlabel("s"); plt.title(title); plt.colorbar(label="dB")
    plt.tight_layout(); plt.show()

---
### 🕐 Session 1 of 3 — *Reading Spectrograms* (~35 min)
**Goal:** learn to sight-read time–frequency pictures: tones, chirps, harmonics, percussion.
**Builds on:** [DSP Foundations](./Foundations_of_Signal_Processing_1.ipynb) S6. &nbsp; **Feeds into:** Session 2 (speech).

---

## 2. The Spectrogram as Sheet Music

💡 **Intuition.** A spectrogram is *sheet music extracted from sound*: time runs right, pitch runs up, ink is energy. Horizontal lines = steady tones; stacked lines = harmonics of one note (their spacing IS the pitch); vertical stripes = clicks/percussion (uncertainty principle: sharp in time ⇒ smeared in frequency); sweeps = chirps. Learn these four glyphs and you can 'read' most sounds before hearing them.

In [2]:
t = np.arange(0, 2.5, 1/fs)
melody = np.zeros_like(t)
# three notes with harmonics (a mini 'instrument')
for start, f0 in [(0.1, 220), (0.9, 277), (1.7, 330)]:
    seg = (t >= start) & (t < start + 0.7)
    for h, amp in [(1, 1.0), (2, 0.5), (3, 0.25), (4, 0.12)]:
        melody[seg] += amp * np.sin(2*np.pi*h*f0*t[seg])
    melody[seg] *= np.hanning(seg.sum())            # note envelope
# percussion: two clicks
for click_t in [0.5, 1.3]:
    idx = int(click_t * fs)
    melody[idx:idx+80] += 2.0 * rng.standard_normal(80) * np.exp(-np.arange(80)/20)

show_spec(melody, "read it: 3 harmonic notes (A3, C#4, E4) + 2 clicks")

/tmp/ipykernel_2030389/3734263403.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


---
### 🕐 Session 2 of 3 — *Speech: the Source-Filter Model* (~40 min)
**Goal:** synthesize vowels from scratch; estimate pitch and formants from a signal.
**Builds on:** Session 1. &nbsp; **Feeds into:** Session 3 (effects).

---

## 3. How Speech Works

💡 **Intuition.** Speech is a two-stage instrument: the **source** (vocal cords buzzing at the pitch $f_0$, or noise for whispers/fricatives) drives a **filter** (the vocal tract, whose resonances — *formants* — are shaped by your tongue and lips). Vowels are *filter settings*: /a/ vs /i/ differ in formant positions, not pitch. This source-filter factorization is the basis of vocoders, LPC compression (your phone), and autotune.

In [3]:
def vowel(f0, formants, dur=0.6, fs=fs):
    """Synthesize a vowel: glottal pulse train through formant resonators."""
    n_samp = int(dur * fs)
    src = np.zeros(n_samp)
    src[::int(fs/f0)] = 1.0                          # impulse train at the pitch
    x = src.copy()
    for fc, bw in formants:                          # cascade of resonators (poles!)
        r = np.exp(-np.pi * bw / fs)
        theta = 2*np.pi*fc/fs
        x = sig.lfilter([1], [1, -2*r*np.cos(theta), r**2], x)
    return x / np.abs(x).max()

# classic formant tables: /a/ (father) vs /i/ (see)
a_sound = vowel(120, [(730, 90), (1090, 110), (2440, 170)])
i_sound = vowel(120, [(270, 60), (2290, 100), (3010, 180)])
both = np.concatenate([a_sound, np.zeros(1600), i_sound])
show_spec(both, "synthetic /a/ then /i/: same pitch (harmonic spacing), different formants (bright bands)")

/tmp/ipykernel_2030389/3734263403.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


In [4]:
# Pitch estimation by autocorrelation: the lag where the signal rhymes with itself
def estimate_pitch(x, fs=fs, fmin=60, fmax=400):
    x = x - x.mean()
    r = np.correlate(x, x, "full")[len(x)-1:]
    lo, hi = int(fs/fmax), int(fs/fmin)
    return fs / (lo + np.argmax(r[lo:hi]))

print(f"estimated pitch of /a/: {estimate_pitch(a_sound):.1f} Hz  (synthesized at 120 Hz)")

# Formant estimation via LPC (all-pole fit — Wiener/Yule-Walker in disguise)
def lpc_formants(x, order=10, fs=fs):
    x = x * np.hanning(len(x))
    r = np.correlate(x, x, "full")[len(x)-1:len(x)+order]
    from scipy.linalg import solve_toeplitz
    a = solve_toeplitz(r[:-1], r[1:])                # Yule-Walker
    roots = np.roots(np.concatenate([[1], -a]))
    roots = roots[(roots.imag > 0)]
    freqs = np.sort(np.angle(roots) * fs / (2*np.pi))
    return freqs[freqs > 90][:3]

print("estimated /a/ formants:", np.round(lpc_formants(a_sound)), " (target ≈ [730, 1090, 2440])")
print("estimated /i/ formants:", np.round(lpc_formants(i_sound)), " (target ≈ [270, 2290, 3010])")

estimated pitch of /a/: 120.3 Hz  (synthesized at 120 Hz)
estimated /a/ formants: [ 723. 1088. 2365.]  (target ≈ [730, 1090, 2440])
estimated /i/ formants: [ 257. 2288. 3010.]  (target ≈ [270, 2290, 3010])


---
### 🕐 Session 3 of 3 — *Effects Are Filters* (~35 min)
**Goal:** build reverb, robot voice, and a pitch shifter — and see each as a DSP primitive.
**Builds on:** Session 2.

---

## 4. The Effects Rack

Every studio effect is a signal-processing primitive wearing a costume:

| Effect | DSP primitive |
|---|---|
| Echo/reverb | convolution with a (sparse/dense) impulse response |
| Robot voice | ring modulation (multiply by a carrier) |
| Wah / EQ | time-varying / fixed [filters](./Filter_Design.ipynb) |
| Pitch shift | STFT: stretch time, then resample ([multirate](./Foundations_of_Signal_Processing_2.ipynb)) |

In [5]:
voice = both                                        # our /a/-/i/ 'phrase'

# 1) Reverb: convolve with an exponentially decaying random impulse response
ir = rng.standard_normal(int(0.4*fs)) * np.exp(-np.arange(int(0.4*fs)) / (0.12*fs))
ir[0] = 1.0
reverbed = sig.fftconvolve(voice, 0.4*ir)[:len(voice)]

# 2) Robot: ring-modulate with a 70 Hz carrier
robot = voice * np.sin(2*np.pi*70*np.arange(len(voice))/fs)

# 3) Pitch shift up a fourth: phase-vocoder-lite = time-stretch (STFT hop trick) + resample
f_, t_, S = sig.stft(voice, fs=fs, nperseg=1024, noverlap=768)
_, stretched = sig.istft(S, fs=fs, nperseg=1024, noverlap=896)     # smaller synthesis hop → slower
shifted = sig.resample(stretched, len(voice))                       # resample back → higher pitch

fig, axes = plt.subplots(1, 3, figsize=(10.5, 2.5))
for ax, (y, name) in zip(axes, [(reverbed, "reverb"), (robot, "robot (sidebands!)"), (shifted, "pitch-shifted")]):
    fq, tq, Sq = sig.stft(y, fs=fs, nperseg=512)
    ax.pcolormesh(tq, fq, 20*np.log10(np.abs(Sq)+1e-8), shading="auto", vmin=-100, vmax=-20)
    ax.set_title(name); ax.set_ylim(0, 4000)
plt.tight_layout(); plt.show()
print("export any of these:  from scipy.io import wavfile;  wavfile.write('out.wav', fs, (y*32767).astype('int16'))")

export any of these:  from scipy.io import wavfile;  wavfile.write('out.wav', fs, (y*32767).astype('int16'))


/tmp/ipykernel_2030389/2223069117.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


## 5. Conclusion

Spectrograms are readable sheet music; speech factors into source × filter (pitch × formants); and the entire effects industry is convolution, modulation, filtering, and resampling with good marketing. Record a real voice and rerun every cell — that's the homework.

---
## Where next

- [CNN workshop](../Intro_Mach_Learn/Intro_CNN/Intro_CNN.ipynb) — classifying exactly these spectrograms.
- [Foundations 2](./Foundations_of_Signal_Processing_2.ipynb) — the multirate machinery under the pitch shifter.
- [Array Processing](./Array_Processing.ipynb) — what the *second* microphone buys you.